<a href="https://colab.research.google.com/github/OdysseusPolymetis/tragic_influence_Litterature/blob/main/tragic_influence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers
!git clone https://github.com/diyclassics/spacy-lookups-data.git
!pip install -r /content/spacy-lookups-data/requirements.txt
!python -m pytest /content/spacy-lookups-data/spacy_lookups_data
!pip install "la-core-web-lg @ https://huggingface.co/latincy/la_core_web_lg/resolve/main/la_core_web_lg-any-py3-none-any.whl"
!git clone https://github.com/PerseusDL/treebank_data.git
!git clone https://github.com/PerseusDL/canonical-latinLit.git
!git clone https://github.com/PerseusDL/canonical-greekLit.git
!pip install stanza

In [ ]:
import os
import re
import unicodedata
from collections import defaultdict, Counter
import random

import numpy as np
import torch
import torch.nn.functional as F
import seaborn as sns
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch

from scipy.stats import chi2_contingency
from lxml import etree
from tqdm import tqdm
import spacy
import stanza
from sentence_transformers import SentenceTransformer

In [ ]:
stanza.download('grc')
nlp_greek = stanza.Pipeline('grc', processors='tokenize,pos')

# 0. Définition du corpus

In [ ]:
greek_tragedians = {
    'sophocles': ['tlg0011'],
    'aeschylus': ['tlg0085'],
    'euripides': ['tlg0006']
}

# Corpus de référence dramatique (pour contraste intra-genre)
dramatic_reference = {
    'comedy': [
        'tlg0019',
        'tlg0039',
    ],
    'other_tragedy': [
        # Ajouter fragments d'autres tragiques si disponibles
        # Par exemple les tragiques mineurs
    ]
}

# Corpus de référence non-dramatique (pour contraste inter-genre)
non_dramatic_reference = {
    'epic': ['tlg0012', 'tlg0020'],      # Homère, Hésiode
    'prose': ['tlg0016', 'tlg0003'],     # Hérodote, Thucydide
    'philosophy': ['tlg0059', 'tlg0086']  # Platon, Aristote
}

# Corpus latin cible - tragédies de Sénèque uniquement
seneca_tragedies = {
    'phi001': 'Hercules Furens',
    'phi002': 'Troades',
    'phi003': 'Phoenissae',
    'phi004': 'Medea',
    'phi005': 'Phaedra',
    'phi006': 'Oedipus',
    'phi007': 'Agamemnon',
    'phi008': 'Thyestes',
    'phi009': 'Hercules Oetaeus',
    'phi010': 'Octavia'
}

print("Corpus définis:")
print(f"  Tragiques grecs à analyser: {list(greek_tragedians.keys())}")
print(f"  Référence dramatique: {list(dramatic_reference.keys())}")
print(f"  Référence non-dramatique: {list(non_dramatic_reference.keys())}")
print(f"  Tragédies de Sénèque: {len(seneca_tragedies)} œuvres")

# 1. Préparation des fonctions de nettoyage et de segmentation

In [ ]:
nlp = spacy.load('la_core_web_lg', disable=["parser", "ner", "lemmatizer", "textcat"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def normalize_text(text):
    # Normaliser (NFC), toLower
    text = unicodedata.normalize('NFC', text)
    return text.lower()

In [ ]:
def clean_segment(segment):
    # Supprimer le contenu entre crochets
    segment = re.sub(r'\[.*?\]', '', segment)
    # Supprimer les références abrégées (ex : adv., math., etc.)
    segment = re.sub(r'\b(?:adv|math|vgl|phil|of|vol|p)\.\b', '', segment, flags=re.IGNORECASE)
    # Supprimer les chiffres
    segment = re.sub(r'\d+', '', segment)
    # Supprimer la ponctuation non pertinente
    segment = re.sub(r'[^\w\s]', '', segment)
    # Remplacer les suites d'espaces par un seul espace
    segment = re.sub(r'\s+', ' ', segment)
    return segment.strip()

In [ ]:
def segment_latin_text_multiscale(text, max_batch_size=2048, min_length=3, file_info=None):
    """Segmente un texte latin en trois niveaux de granularité"""
    segments = {'lexical': [], 'thematic': [], 'structural': []}
    text_batches = [text[i:i + max_batch_size] for i in range(0, len(text), max_batch_size)]
    unit_segments = []
    current_segment = []

    if file_info and len(text_batches) > 5:
        print(f"  Segmentation de {file_info} ({len(text_batches)} batches)...", end='', flush=True)

    for batch in text_batches:
        doc = nlp(batch)
        for token in doc:
            current_segment.append(token.text)
            if token.pos_ in ['SCONJ', 'CCONJ', 'PUNCT']:
                if len(current_segment) >= min_length:
                    unit_segments.append(' '.join(current_segment))
                current_segment = []
        if len(current_segment) >= min_length:
            unit_segments.append(' '.join(current_segment))
            current_segment = []

    if file_info and len(text_batches) > 5:
        print(" ✓")

    segments['lexical'] = unit_segments

    for i in range(len(unit_segments) - 2):
        group = ' '.join(unit_segments[i:i+3])
        if len(group.split()) >= 7:
            segments['thematic'].append(group)

    for i in range(len(unit_segments) - 7):
        group = ' '.join(unit_segments[i:i+8])
        if len(group.split()) >= 20:
            segments['structural'].append(group)

    return segments

In [ ]:
def segment_greek_text(tree, min_length=3):
    """Segmente un TEI tree grec via postag"""
    segments = []
    current_segment = []

    for word in tree.findall(".//word"):
        form = word.get("form")
        postag = word.get("postag")
        if form:
            current_segment.append(form)
            if postag and postag[0].lower() in ['c', 'u']:
                if len(current_segment) >= min_length:
                    segments.append(' '.join(current_segment))
                current_segment = []

    if len(current_segment) >= min_length:
        segments.append(' '.join(current_segment))

    return segments

In [ ]:
def segment_greek_text_stanza(text, min_length=3):
    """Segmente un texte grec brut via Stanza"""
    segments = []
    current_segment = []
    doc = nlp_greek(text)

    for sentence in doc.sentences:
        for token in sentence.tokens:
            current_segment.append(token.text)
            if token.words[0].upos in ['CCONJ', 'SCONJ', 'PUNCT']:
                if len(current_segment) >= min_length:
                    segments.append(' '.join(current_segment))
                current_segment = []

    if len(current_segment) >= min_length:
        segments.append(' '.join(current_segment))

    return segments

# 2. Extraction des XML

In [ ]:
def extract_body_content_from_xml(file_path):
    """Extrait et segmente le contenu XML selon la langue"""
    parser = etree.XMLParser(recover=True)
    tree = etree.parse(file_path, parser)
    nsmap = tree.getroot().nsmap
    default_ns = nsmap.get(None)

    body = tree.find(f".//{{{default_ns}}}body" if default_ns else ".//body")
    if body is None:
        raise ValueError(f"No <body> element found in {file_path}")

    text = ''.join(body.itertext())

    if 'grc' in file_path or 'greek' in file_path.lower():
        if tree.find(".//word") is not None:
            return segment_greek_text(tree)
        else:
            return segment_greek_text_stanza(text)
    else:
        file_name = os.path.basename(file_path).split('.')[0]
        return segment_latin_text_multiscale(text, file_info=file_name)

In [ ]:
def extract_texts_from_directory(directory_path, target_authors, lang='la'):
    """Extrait les textes selon le type de corpus"""
    texts = {}
    files_to_process = []

    if 'canonical-greekLit' in directory_path:
        for author in target_authors:
            author_path = os.path.join(directory_path, 'data', author)
            if os.path.exists(author_path):
                for work_dir in os.listdir(author_path):
                    work_path = os.path.join(author_path, work_dir)
                    if os.path.isdir(work_path):
                        for file in os.listdir(work_path):
                            if file.endswith('.xml') and 'grc' in file and 'eng' not in file:
                                files_to_process.append(os.path.join(work_path, file))

    elif 'canonical-latinLit' in directory_path:
        for root, _, files in os.walk(directory_path):
            for file in files:
                if file.endswith('.xml') and 'lat' in file and any(author in file for author in target_authors):
                    files_to_process.append(os.path.join(root, file))

    # Traitement unifié avec barre de progression
    desc = f"Extraction textes {'grecs' if lang == 'grc' else 'latins'}"
    for file_path in tqdm(files_to_process, desc=desc):
        try:
            segments = extract_body_content_from_xml(file_path)
            file_name = os.path.basename(file_path)

            if lang == 'grc':
                cleaned_segments = []
                for s in segments:
                    cleaned = clean_segment(s)
                    if (cleaned and len(cleaned.split()) >= 5 and
                        not any(c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz' for c in cleaned[:20])):
                        cleaned_segments.append(cleaned)
                texts[file_name] = cleaned_segments
            else:
                cleaned_segments_dict = {}
                for level, segs in segments.items():
                    cleaned_segments_dict[level] = [clean_segment(s) for s in segs
                                                   if len(clean_segment(s).split()) >= 5]
                texts[file_name] = cleaned_segments_dict

        except Exception as e:
            print(f"\nErreur avec {os.path.basename(file_path)}: {e}")

    return texts

#4. Extraction des corpus

Selon le nombre de textes inclus dans le corpus de référence, cette partie peut prendre du temps. Ici, par défaut, cela fait à peu près 5 minutes.

In [ ]:
print("1. Chargement des tragiques grecs...")
tragedian_corpora = {}
for name, codes in greek_tragedians.items():
    corpus = extract_texts_from_directory('/content/canonical-greekLit', codes, lang='grc')
    tragedian_corpora[name] = corpus
    print(f"  {name}: {len(corpus)} fichiers")

# 2. Référence dramatique
print("\n2. Corpus dramatique de référence...")
dramatic_corpus = {}
for genre, codes in dramatic_reference.items():
    if codes:
        corpus = extract_texts_from_directory('/content/canonical-greekLit', codes, lang='grc')
        dramatic_corpus[genre] = corpus
        print(f"  {genre}: {len(corpus)} fichiers")

# 3. Référence non-dramatique
print("\n3. Corpus non-dramatique de référence...")
non_dramatic_corpus = {}
for genre, codes in non_dramatic_reference.items():
    corpus = extract_texts_from_directory('/content/canonical-greekLit', codes, lang='grc')
    non_dramatic_corpus[genre] = corpus
    print(f"  {genre}: {len(corpus)} fichiers")

# 4. Sénèque
print("\n4. Tragédies de Sénèque...")
lat_seneca = ["phi1017"]  # Code auteur de Sénèque, pas les codes d'œuvres
seneca_corpus = extract_texts_from_directory('/content/canonical-latinLit',
                                            lat_seneca, lang='la')
print(f"  Sénèque: {len(seneca_corpus)} fichiers")

In [ ]:
print("\n=== CRÉATION DES FENÊTRES ===\n")

# 1. Fenêtres pour les tragiques grecs
tragedian_windows = {}
for name, corpus in tragedian_corpora.items():
    windows = [(seg, file_id) for file_id, segs in corpus.items()
               for seg in segs if seg.strip()]
    tragedian_windows[name] = windows
    print(f"{name}: {len(windows)} segments")

# 2. Fenêtres pour la référence dramatique
dramatic_windows = []
for genre, corpus in dramatic_corpus.items():
    for file_id, segs in corpus.items():
        for seg in segs:
            if seg.strip():
                dramatic_windows.append((seg, file_id))
print(f"\nRéférence dramatique: {len(dramatic_windows)} segments")

# 3. Fenêtres pour la référence non-dramatique
non_dramatic_windows = []
for genre, corpus in non_dramatic_corpus.items():
    for file_id, segs in corpus.items():
        for seg in segs:
            if seg.strip():
                non_dramatic_windows.append((seg, file_id))
print(f"Référence non-dramatique: {len(non_dramatic_windows)} segments")

# 4. Fenêtres Sénèque multi-échelle
seneca_windows = {'lexical': [], 'thematic': [], 'structural': []}
for file_id, levels_dict in seneca_corpus.items():
    for level, segments in levels_dict.items():
        for seg in segments:
            if seg.strip():
                seneca_windows[level].append((seg, file_id))

print(f"\nSénèque:")
for level in ['lexical', 'thematic', 'structural']:
    print(f"  {level}: {len(seneca_windows[level])} segments")

# FILTRAGE : Garder uniquement les vraies tragédies de Sénèque
print("\n=== FILTRAGE DES VRAIES TRAGÉDIES DE SÉNÈQUE ===")

# Définir les codes des vraies tragédies (exclure phi010 et au-delà)
tragedy_codes = ['phi001', 'phi002', 'phi003', 'phi004', 'phi005',
                 'phi006', 'phi007', 'phi008', 'phi009']

# Créer de nouvelles fenêtres filtrées
seneca_tragedy_windows = {'lexical': [], 'thematic': [], 'structural': []}

for level in ['lexical', 'thematic', 'structural']:
    for seg, file_id in seneca_windows[level]:
        # Vérifier si c'est une vraie tragédie
        if any(code in file_id for code in tragedy_codes):
            seneca_tragedy_windows[level].append((seg, file_id))
    print(f"  {level} (tragédies seulement): {len(seneca_tragedy_windows[level])} segments")

# Remplacer les fenêtres originales par les filtrées
seneca_windows = seneca_tragedy_windows

# 5. Filtrer Euripide (9 œuvres principales)
euripides_selected = ['tlg003', 'tlg005', 'tlg017', 'tlg002',
                      'tlg011', 'tlg008', 'tlg016', 'tlg012', 'tlg006']
euripides_filtered = [(seg, fid) for seg, fid in tragedian_windows['euripides']
                      if any(work in fid for work in euripides_selected)]
tragedian_windows['euripides'] = euripides_filtered
print(f"\nEuripide après filtrage: {len(euripides_filtered)} segments")

# 5. Calcul des LLR

In [ ]:
def calculate_double_specificity(author_name, author_windows, other_tragedians_windows,
                                 dramatic_windows, non_dramatic_windows,
                                 min_word_length=4, min_frequency=3):
    """
    Calcule la spécificité à deux niveaux :
    1. vs autres tragiques + comédie (spécificité dans le genre dramatique)
    2. vs corpus non-dramatique (spécificité du genre dramatique)
    """
    # Vocabulaire de l'auteur
    author_vocab = Counter()
    for seg, _ in author_windows:
        for word in seg.split():
            if len(word) >= min_word_length and not word[0].isupper():
                author_vocab[word] += 1

    # Vocabulaire dramatique (autres tragiques + comédie)
    dramatic_vocab = Counter()
    for seg, _ in other_tragedians_windows + dramatic_windows:
        for word in seg.split():
            if len(word) >= min_word_length and not word[0].isupper():
                dramatic_vocab[word] += 1

    # Vocabulaire non-dramatique
    non_dramatic_vocab = Counter()
    for seg, _ in non_dramatic_windows:
        for word in seg.split():
            if len(word) >= min_word_length and not word[0].isupper():
                non_dramatic_vocab[word] += 1

    signatures = []
    for word, freq in author_vocab.most_common(1000):
        if freq >= min_frequency:
            # Score 1 : spécificité dans le genre dramatique
            dramatic_freq = dramatic_vocab.get(word, 0)
            if dramatic_freq == 0:
                dramatic_score = freq * 2  # Bonus si absent du reste du théâtre
            else:
                dramatic_score = freq / (dramatic_freq + 1)

            # Score 2 : spécificité vs non-dramatique
            non_dramatic_freq = non_dramatic_vocab.get(word, 0)
            if non_dramatic_freq > 100:  # Trop commun dans le corpus général
                continue
            elif non_dramatic_freq == 0:
                genre_score = 1.5  # Fort bonus si spécifique au théâtre
            else:
                genre_score = 1.0 / (1 + non_dramatic_freq/10)

            # Score combiné
            combined_score = dramatic_score * genre_score

            if dramatic_score > 1.5:  # Doit être spécifique à l'auteur
                signatures.append({
                    'word': word,
                    'freq': freq,
                    'dramatic_score': dramatic_score,
                    'genre_score': genre_score,
                    'combined_score': combined_score
                })

    signatures.sort(key=lambda x: x['combined_score'], reverse=True)
    return signatures[:100]  # Top 100

# Calculer pour chaque tragique
print("=== CALCUL DES SIGNATURES À DOUBLE NIVEAU ===\n")

author_signatures_double = {}

for name in ['sophocles', 'aeschylus', 'euripides']:
    print(f"Calcul pour {name}...")

    # Fenêtres des autres tragiques
    other_tragedians = []
    for other_name, windows in tragedian_windows.items():
        if other_name != name:
            other_tragedians.extend(windows)

    signatures = calculate_double_specificity(
        name,
        tragedian_windows[name],
        other_tragedians,
        dramatic_windows,
        non_dramatic_windows
    )

    author_signatures_double[name] = signatures

    print(f"  Top 5 signatures de {name}:")
    for sig in signatures[:5]:
        print(f"    {sig['word']}: score={sig['combined_score']:.2f}")
        print(f"      (dramatique={sig['dramatic_score']:.2f}, genre={sig['genre_score']:.2f})")

print("\nSignatures calculées pour les 3 tragiques")

In [ ]:
def select_representative_segments_v2(author_name, author_windows, signatures, n_segments=300):
    """Sélectionne les segments les plus riches en vocabulaire signature"""
    signature_words = set([item['word'] for item in signatures[:50]])

    segment_scores = []
    for seg, file_id in author_windows:
        words = seg.split()
        signature_count = sum(1 for w in words if w in signature_words)
        if len(words) > 0:
            density = signature_count / len(words)
        else:
            density = 0

        segment_scores.append({
            'segment': seg,
            'file_id': file_id,
            'density': density,
            'count': signature_count
        })

    segment_scores.sort(key=lambda x: x['density'], reverse=True)
    return segment_scores[:n_segments]

# Sélectionner les segments
print("\n=== SÉLECTION DES SEGMENTS REPRÉSENTATIFS ===\n")

representative_segments = {}
for name in ['sophocles', 'aeschylus', 'euripides']:
    segments = select_representative_segments_v2(
        name,
        tragedian_windows[name],
        author_signatures_double[name]
    )
    representative_segments[name] = segments
    print(f"{name}: {len(segments)} segments sélectionnés")
    print(f"  Densité moyenne: {np.mean([s['density'] for s in segments]):.3f}")

# Encoder avec SPhilBerta
print("\n=== ENCODAGE DES SEGMENTS ===\n")

greek_embeddings = {}
for name in ['sophocles', 'aeschylus', 'euripides']:
    texts = [s['segment'] for s in representative_segments[name]]
    print(f"Encodage {name}...")
    embeddings = compute_embeddings_sphilberta(texts)
    greek_embeddings[name] = embeddings
    print(f"  Shape: {embeddings.shape}")

# Encoder Sénèque
print("\nEncodage Sénèque...")
seneca_embeddings = {}
for level in ['lexical', 'thematic', 'structural']:
    texts = [seg for seg, _ in seneca_windows[level]]
    print(f"  Niveau {level}...")
    embeddings = compute_embeddings_sphilberta(texts)
    seneca_embeddings[level] = embeddings
    print(f"    Shape: {embeddings.shape}")

# 7. Encodage avec le transformer

In [ ]:
model = SentenceTransformer("bowphs/SPhilBerta").to(device)

In [ ]:
def compute_embeddings_sphilberta(text_windows, batch_size=512):
    """
    Encode les fenêtres par batch et renvoie un NumPy array d'embeddings.
    On utilise un cache dict pour éviter de re-calculer l'embedding de la même
    phrase plusieurs fois si elle apparaît plusieurs fois.
    """
    cache = {}
    embeddings = []
    to_encode = []

    # On construit une liste des fenêtres à encoder, tout en vérifiant le cache
    for tw in text_windows:
        if tw not in cache:
            to_encode.append(tw)

    # Encodage par batch
    with torch.no_grad():
        for i in tqdm(range(0, len(to_encode), batch_size), desc="Computing embeddings"):
            batch = to_encode[i:i + batch_size]
            batch_embeddings = model.encode(batch, convert_to_tensor=True, device=device)
            # On stocke dans le cache
            for text, emb in zip(batch, batch_embeddings):
                cache[text] = emb.cpu().numpy()

    # Maintenant, on produit la liste finale des embeddings dans l'ordre des `text_windows`
    for tw in text_windows:
        embeddings.append(cache[tw])

    # On retourne un array (N, D)
    return np.vstack(embeddings)

In [ ]:
def find_semantic_echoes(greek_embeddings, latin_embeddings, threshold=0.75, top_k=5):
    """Version actuelle que vous utilisez"""
    similarities = np.dot(greek_embeddings, latin_embeddings.T) / (
        np.linalg.norm(greek_embeddings, axis=1)[:, np.newaxis] *
        np.linalg.norm(latin_embeddings, axis=1)[np.newaxis, :]
    )

    echoes = []
    for i in range(len(greek_embeddings)):
        top_indices = np.argsort(similarities[i])[-top_k:][::-1]
        top_scores = similarities[i][top_indices]

        valid = top_scores >= threshold
        if np.any(valid):
            echoes.append({
                'greek_idx': i,
                'latin_indices': top_indices[valid],
                'scores': top_scores[valid]
            })

    return echoes, similarities

In [ ]:
def get_adaptive_threshold(segment_length, base_threshold=0.75, level='lexical'):
    """
    Ajuste le seuil selon la longueur du segment
    Segments courts → seuil plus élevé
    Segments longs → seuil plus bas
    """
    if level == 'lexical':
        if segment_length < 15:
            return base_threshold + 0.08
        elif segment_length < 30:
            return base_threshold + 0.04
        else:
            return base_threshold
    elif level == 'thematic':
        if segment_length < 30:
            return base_threshold + 0.05
        elif segment_length < 50:
            return base_threshold
        else:
            return base_threshold - 0.03
    else:  # structural
        if segment_length < 50:
            return base_threshold + 0.03
        else:
            return base_threshold - 0.05

In [ ]:
def analyze_length_similarity_correlation(results_adjusted, representative_segments, seneca_windows):
    """Analyse la corrélation entre longueur des segments et scores de similarité"""
    print("\n=== ANALYSE CORRÉLATION LONGUEUR-SIMILARITÉ ===")

    for level in ['lexical', 'thematic', 'structural']:
        lengths = []
        scores = []

        for author in ['sophocles', 'aeschylus', 'euripides']:
            echoes = results_adjusted[author][level]['echoes']
            for echo in echoes:
                greek_idx = echo['greek_idx']
                greek_text = representative_segments[author][greek_idx]['segment']

                for latin_idx, score in zip(echo['latin_indices'], echo['scores']):
                    latin_text = seneca_windows[level][latin_idx][0]
                    # Longueur combinée des deux segments
                    combined_length = len(greek_text.split()) + len(latin_text.split())
                    lengths.append(combined_length)
                    scores.append(score)

        # Corrélation
        if lengths:
            correlation = np.corrcoef(lengths, scores)[0,1]
            print(f"\n{level.upper()}: corrélation = {correlation:.3f}")

            # Visualiser par bandes de longueur
            bins = [0, 20, 40, 60, 100, 200]
            for i in range(len(bins)-1):
                mask = (np.array(lengths) >= bins[i]) & (np.array(lengths) < bins[i+1])
                if np.any(mask):
                    mean_score = np.mean(np.array(scores)[mask])
                    count = np.sum(mask)
                    print(f"  Longueur {bins[i]:3d}-{bins[i+1]:3d} mots: score moyen = {mean_score:.3f} (n={count})")


In [ ]:
def find_semantic_echoes_adaptive(greek_embeddings, latin_embeddings,
                                  greek_texts, latin_texts,
                                  base_threshold=0.75, top_k=5, level='lexical'):
    """Version avec seuils adaptatifs selon la longueur"""
    similarities = np.dot(greek_embeddings, latin_embeddings.T) / (
        np.linalg.norm(greek_embeddings, axis=1)[:, np.newaxis] *
        np.linalg.norm(latin_embeddings, axis=1)[np.newaxis, :]
    )

    echoes = []
    for i in range(len(greek_embeddings)):
        greek_length = len(greek_texts[i].split())
        echo_data = {
            'greek_idx': i,
            'latin_indices': [],
            'scores': [],
            'weighted_scores': [],
            'lengths': []
        }

        # Sélectionner les top_k candidats
        top_indices = np.argsort(similarities[i])[-top_k:][::-1]

        for j in top_indices:
            latin_length = len(latin_texts[j].split())
            combined_length = greek_length + latin_length

            # Seuil adaptatif
            threshold = get_adaptive_threshold(combined_length, base_threshold, level)

            if similarities[i, j] >= threshold:
                # Score pondéré par la longueur
                if combined_length < 20:
                    length_weight = 0.8
                elif combined_length < 40:
                    length_weight = 1.0
                elif combined_length < 80:
                    length_weight = 1.15
                else:
                    length_weight = 1.25

                weighted_score = similarities[i, j] * length_weight

                echo_data['latin_indices'].append(j)
                echo_data['scores'].append(similarities[i, j])
                echo_data['weighted_scores'].append(weighted_score)
                echo_data['lengths'].append((greek_length, latin_length))

        if len(echo_data['latin_indices']) > 0:
            echo_data['latin_indices'] = np.array(echo_data['latin_indices'])
            echo_data['scores'] = np.array(echo_data['scores'])
            echo_data['weighted_scores'] = np.array(echo_data['weighted_scores'])
            echoes.append(echo_data)

    return echoes, similarities

In [ ]:
print("\n=== ANALYSE AVEC PARAMÈTRES ADAPTATIFS ===\n")

# D'abord faire l'analyse standard
results_adjusted = {}
for greek_author in ['sophocles', 'aeschylus', 'euripides']:
    results_adjusted[greek_author] = {}

    for level in ['lexical', 'thematic', 'structural']:
        # Paramètres ajustés selon le niveau
        if level == 'lexical':
            threshold = 0.82
            top_k = 5
        elif level == 'thematic':
            threshold = 0.78
            top_k = 10
        else:  # structural
            threshold = 0.73
            top_k = 15

        echoes, sim_matrix = find_semantic_echoes(
            greek_embeddings[greek_author],
            seneca_embeddings[level],
            threshold=threshold,
            top_k=top_k
        )

        results_adjusted[greek_author][level] = {
            'echoes': echoes,
            'echo_count': len(echoes),
            'avg_similarity': np.mean([e['scores'].mean() for e in echoes]) if echoes else 0
        }

        print(f"{greek_author} → Sénèque {level} (seuil={threshold}):")
        print(f"  Échos: {len(echoes)}/300")
        print(f"  Similarité moyenne: {results_adjusted[greek_author][level]['avg_similarity']:.3f}")

# Analyser la corrélation longueur-similarité
analyze_length_similarity_correlation(results_adjusted, representative_segments, seneca_windows)

# Maintenant refaire avec seuils adaptatifs
print("\n=== ANALYSE AVEC SEUILS ADAPTATIFS ===\n")

results_adaptive = {}
for greek_author in ['sophocles', 'aeschylus', 'euripides']:
    results_adaptive[greek_author] = {}
    greek_texts = [s['segment'] for s in representative_segments[greek_author]]

    for level in ['lexical', 'thematic', 'structural']:
        latin_texts = [seg for seg, _ in seneca_windows[level]]

        # Base thresholds
        if level == 'lexical':
            base_threshold = 0.78
            top_k = 5
        elif level == 'thematic':
            base_threshold = 0.74
            top_k = 10
        else:
            base_threshold = 0.68
            top_k = 15

        echoes, sim_matrix = find_semantic_echoes_adaptive(
            greek_embeddings[greek_author],
            seneca_embeddings[level],
            greek_texts,
            latin_texts,
            base_threshold=base_threshold,
            top_k=top_k,
            level=level
        )

        results_adaptive[greek_author][level] = {
            'echoes': echoes,
            'echo_count': len(echoes),
            'avg_similarity': np.mean([e['scores'].mean() for e in echoes]) if echoes else 0,
            'avg_weighted': np.mean([e['weighted_scores'].mean() for e in echoes]) if echoes else 0
        }

        print(f"{greek_author} → Sénèque {level} (adaptatif base={base_threshold}):")
        print(f"  Échos: {len(echoes)}/300")
        print(f"  Similarité moyenne: {results_adaptive[greek_author][level]['avg_similarity']:.3f}")
        print(f"  Score pondéré moyen: {results_adaptive[greek_author][level]['avg_weighted']:.3f}")

# Comparer standard vs adaptatif
print("\n=== COMPARAISON STANDARD vs ADAPTATIF ===")
for level in ['lexical', 'thematic', 'structural']:
    print(f"\n{level.upper()}:")
    print("           Standard  Adaptatif  Différence")
    for author in ['sophocles', 'aeschylus', 'euripides']:
        std_count = results_adjusted[author][level]['echo_count']
        adp_count = results_adaptive[author][level]['echo_count']
        std_sim = results_adjusted[author][level]['avg_similarity']
        adp_sim = results_adaptive[author][level]['avg_similarity']
        print(f"  {author:10} {std_count:3d}/{std_sim:.3f}  {adp_count:3d}/{adp_sim:.3f}  {adp_count-std_count:+3d}/{adp_sim-std_sim:+.3f}")


In [ ]:
def analyze_by_similarity_bands(results_adjusted):
    bands = [(0.85, 1.0, 'Très forte'),
             (0.80, 0.85, 'Forte'),
             (0.75, 0.80, 'Modérée'),
             (0.70, 0.75, 'Faible')]

    for level in ['lexical', 'thematic', 'structural']:
        print(f"\n{level.upper()} - Distribution par bandes de similarité:")
        for author in ['sophocles', 'aeschylus', 'euripides']:
            echoes = results_adjusted[author][level]['echoes']
            all_scores = []
            for e in echoes:
                all_scores.extend(e['scores'].tolist())

            print(f"\n  {author}:")
            for min_val, max_val, label in bands:
                count = sum(1 for s in all_scores if min_val <= s < max_val)
                pct = (count / len(all_scores) * 100) if all_scores else 0
                print(f"    {label} ({min_val:.2f}-{max_val:.2f}): {count} ({pct:.1f}%)")

analyze_by_similarity_bands(results_adjusted)

In [ ]:
# Tester sur la proportion de scores très forts
from scipy.stats import chi2_contingency

for level in ['lexical', 'thematic', 'structural']:
    print(f"\n{level.upper()} - Test sur les proportions très fortes (>0.85):")

    # Compter les scores >0.85 vs ≤0.85 pour chaque auteur
    contingency = []
    for author in ['sophocles', 'aeschylus', 'euripides']:
        echoes = results_adjusted[author][level]['echoes']
        all_scores = []
        for e in echoes:
            all_scores.extend(e['scores'].tolist())

        very_strong = sum(1 for s in all_scores if s > 0.85)
        other = len(all_scores) - very_strong
        contingency.append([very_strong, other])

    chi2, p_value, dof, expected = chi2_contingency(contingency)
    print(f"  χ² = {chi2:.2f}, p = {p_value:.4f}")

    if p_value < 0.05:
        print("  → Différence significative dans les proportions!")
        for i, author in enumerate(['sophocles', 'aeschylus', 'euripides']):
            print(f"    {author}: {contingency[i][0]}/{sum(contingency[i])} = {contingency[i][0]/sum(contingency[i])*100:.1f}%")

In [ ]:
# 1. Bootstrap pour vérifier la stabilité
def bootstrap_analysis(embeddings, n_iterations=100):
    results = []
    for _ in range(n_iterations):
        # Rééchantillonner avec remplacement
        indices = np.random.choice(len(embeddings), len(embeddings), replace=True)
        sample = embeddings[indices]
        # Calculer les métriques sur l'échantillon
    return np.std(results)  # Variance faible = résultats stables

# 2. Analyse de puissance statistique
from statsmodels.stats.power import FTestAnovaPower
power_analysis = FTestAnovaPower()
power = power_analysis.solve_power(effect_size=0.25, nobs=300, alpha=0.05)
print(f"Puissance statistique: {power:.2f}")
# Si > 0.80, vous aviez assez de données pour détecter des différences

# 3. Taille d'effet (même si non significatif)
def cohens_d(group1, group2):
    return (np.mean(group1) - np.mean(group2)) / np.sqrt((np.var(group1) + np.var(group2))/2)
# 2. TAILLES D'EFFET (Cohen's d)
print("\n=== TAILLES D'EFFET (Cohen's d) ===")
def cohens_d(group1, group2):
    return (np.mean(group1) - np.mean(group2)) / np.sqrt((np.var(group1) + np.var(group2))/2)

for level in ['lexical', 'thematic', 'structural']:
    print(f"\n{level.upper()}:")
    scores = {}
    for author in ['sophocles', 'aeschylus', 'euripides']:
        echoes = results_adjusted[author][level]['echoes']
        scores[author] = [e['scores'].max() for e in echoes]

    # Calculer Cohen's d pour chaque paire
    pairs = [('euripides', 'sophocles'), ('euripides', 'aeschylus'), ('sophocles', 'aeschylus')]
    for p1, p2 in pairs:
        d = cohens_d(scores[p1], scores[p2])
        interpretation = "négligeable" if abs(d) < 0.2 else "petit" if abs(d) < 0.5 else "moyen" if abs(d) < 0.8 else "large"
        print(f"  {p1} vs {p2}: d={d:+.3f} ({interpretation})")

# 3. BOOTSTRAP pour vérifier la stabilité
print("\n=== ANALYSE BOOTSTRAP (stabilité) ===")
def bootstrap_similarity(author, level, n_iterations=100):
    """Teste la stabilité des résultats par rééchantillonnage"""
    mean_sims = []

    for _ in range(n_iterations):
        # Rééchantillonner les embeddings
        n_samples = len(greek_embeddings[author])
        indices = np.random.choice(n_samples, n_samples, replace=True)
        sampled_embeddings = greek_embeddings[author][indices]

        # Recalculer les similarités
        echoes, _ = find_semantic_echoes(
            sampled_embeddings,
            seneca_embeddings[level],
            threshold=0.75,
            top_k=5
        )

        if echoes:
            mean_sim = np.mean([e['scores'].mean() for e in echoes])
            mean_sims.append(mean_sim)

    return np.mean(mean_sims), np.std(mean_sims)

print("\nStabilité des similarités moyennes (100 bootstraps):")
for level in ['lexical']:
    print(f"\n{level.upper()}:")
    for author in ['sophocles', 'aeschylus', 'euripides']:
        mean, std = bootstrap_similarity(author, level, n_iterations=50)
        cv = (std/mean)*100 if mean > 0 else 0
        print(f"  {author}: μ={mean:.4f} ± {std:.4f} (CV={cv:.1f}%)")
    print(f"  → CV < 5% = très stable, CV < 10% = stable")

# 4. VALIDATION CROISÉE
print("\n=== VALIDATION CROISÉE ===")
def cross_validate_influence(author, level, n_folds=5):
    """Validation croisée k-fold"""
    n_samples = len(representative_segments[author])
    fold_size = n_samples // n_folds
    similarities = []

    for fold in range(n_folds):
        # Définir train/test
        test_start = fold * fold_size
        test_end = test_start + fold_size if fold < n_folds-1 else n_samples
        test_indices = list(range(test_start, test_end))

        # Embeddings du fold test
        test_embeddings = greek_embeddings[author][test_indices]

        # Calculer similarités
        echoes, _ = find_semantic_echoes(
            test_embeddings,
            seneca_embeddings[level],
            threshold=0.75,
            top_k=5
        )

        if echoes:
            fold_sim = np.mean([e['scores'].mean() for e in echoes])
            similarities.append(fold_sim)

    return np.mean(similarities), np.std(similarities)

print("Validation croisée 5-fold (niveau lexical):")
for author in ['sophocles', 'aeschylus', 'euripides']:
    mean, std = cross_validate_influence(author, 'lexical')
    print(f"  {author}: {mean:.4f} ± {std:.4f}")

In [ ]:
def analyze_by_seneca_tragedy(results_adjusted, seneca_windows):
    """Analyse quelle tragédie de Sénèque est influencée par quel grec"""

    # Mapping des codes vers noms de tragédies
    tragedy_names = {
        'phi001': 'Hercules Furens',
        'phi002': 'Troades',
        'phi003': 'Phoenissae',
        'phi004': 'Medea',
        'phi005': 'Phaedra',
        'phi006': 'Oedipus',
        'phi007': 'Agamemnon',
        'phi008': 'Thyestes',
        'phi009': 'Hercules Oetaeus'
    }

    # Pour chaque niveau
    for level in ['lexical', 'thematic', 'structural']:
        print(f"\n{'='*60}")
        print(f"NIVEAU {level.upper()}")
        print('='*60)

        # Collecter les influences par tragédie
        tragedy_influences = {name: {'sophocles': [], 'aeschylus': [], 'euripides': []}
                             for name in tragedy_names.values()}

        for greek_author in ['sophocles', 'aeschylus', 'euripides']:
            echoes = results_adjusted[greek_author][level]['echoes']

            for echo in echoes:
                for latin_idx, score in zip(echo['latin_indices'], echo['scores']):
                    file_id = seneca_windows[level][latin_idx][1]

                    # Identifier la tragédie
                    for code, name in tragedy_names.items():
                        if code in file_id:
                            tragedy_influences[name][greek_author].append(score)
                            break

        # Calculer les moyennes et afficher
        for tragedy, influences in tragedy_influences.items():
            scores = []
            for author in ['sophocles', 'aeschylus', 'euripides']:
                if influences[author]:
                    avg = np.mean(influences[author])
                    scores.append((author, avg, len(influences[author])))

            if scores:
                scores.sort(key=lambda x: x[1], reverse=True)
                print(f"\n{tragedy}:")
                for author, avg, count in scores[:3]:
                    print(f"  {author}: {avg:.3f} ({count} échos)")

analyze_by_seneca_tragedy(results_adjusted, seneca_windows)

In [ ]:
greek_works_detail = {}
for greek_author in ['sophocles', 'aeschylus', 'euripides']:
    work_contributions = {}
    for level in ['lexical', 'thematic', 'structural']:
        echoes = results_adjusted[greek_author][level]['echoes']
        for echo in echoes:
            greek_idx = echo['greek_idx']
            greek_file = representative_segments[greek_author][greek_idx]['file_id']
            if '.' in greek_file:
                work_code = greek_file.split('.')[1]
            else:
                work_code = greek_file
            if work_code not in work_contributions:
                work_contributions[work_code] = []
            work_contributions[work_code].extend(echo['scores'].tolist())
    greek_works_detail[greek_author] = work_contributions

# Créer la liste des top œuvres
top_works = []
for author, works in greek_works_detail.items():
    for work_code, scores in works.items():
        if scores:
            top_works.append({
                'author': author,
                'work': work_code,
                'mean_score': np.mean(scores),
                'count': len(scores),
                'label': f"{author[:4].capitalize()}-{work_code}"
            })

top_works.sort(key=lambda x: x['mean_score'], reverse=True)
top_15 = top_works[:15]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def create_influence_heatmap(results_adjusted):
    """Créer une heatmap des influences grec → Sénèque"""

    # Préparer les données
    data = []
    for level in ['lexical', 'thematic', 'structural']:
        row = []
        for author in ['sophocles', 'aeschylus', 'euripides']:
            avg_sim = results_adjusted[author][level]['avg_similarity']
            row.append(avg_sim)
        data.append(row)

    # Créer la heatmap
    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(data,
                annot=True,
                fmt='.3f',
                xticklabels=['Sophocle', 'Eschyle', 'Euripide'],
                yticklabels=['Lexical', 'Thématique', 'Structurel'],
                cmap='YlOrRd',
                cbar_kws={'label': 'Similarité moyenne'},
                vmin=0.78, vmax=0.81)

    plt.title('Influence des tragiques grecs sur Sénèque par niveau')
    plt.tight_layout()
    plt.show()

create_influence_heatmap(results_adjusted)

In [ ]:
def show_highest_similarities_simple(results_adjusted, representative_segments, seneca_windows, n=10):
    """Version simplifiée : montrer juste les textes avec les plus fortes similarités"""

    all_matches = []

    for greek_author in ['sophocles', 'aeschylus', 'euripides']:
        for level in ['lexical', 'thematic', 'structural']:
            echoes = results_adjusted[greek_author][level]['echoes']

            for echo in echoes:
                greek_idx = echo['greek_idx']
                best_latin_idx = echo['latin_indices'][np.argmax(echo['scores'])]
                best_score = echo['scores'].max()

                greek_text = representative_segments[greek_author][greek_idx]['segment']
                latin_text = seneca_windows[level][best_latin_idx][0]
                latin_file = seneca_windows[level][best_latin_idx][1]

                all_matches.append({
                    'score': best_score,
                    'greek_author': greek_author,
                    'level': level,
                    'greek_text': greek_text,
                    'latin_text': latin_text,
                    'latin_file': latin_file
                })

    all_matches.sort(key=lambda x: x['score'], reverse=True)

    print("="*80)
    print("PASSAGES AVEC LES PLUS FORTES SIMILARITÉS")
    print("="*80)

    for i, match in enumerate(all_matches[:n], 1):
        tragedy = match['latin_file'].split('.')[1] if '.' in match['latin_file'] else '?'

        print(f"\n[{i}] Score: {match['score']:.3f} | {match['greek_author'].upper()} → Sénèque {tragedy}")
        print(f"Niveau: {match['level']}")
        print("-"*60)
        print(f"GREC:  {match['greek_text']}")
        print(f"LATIN: {match['latin_text']}")

show_highest_similarities_simple(results_adjusted, representative_segments, seneca_windows, n=10)

In [ ]:
print("=== CONTRÔLE NÉGATIF : CÉSAR ===\n")

# 1. Charger César
caesar_codes = ["phi0448"]  # Code pour César
print("Chargement de César (contrôle négatif)...")
caesar_corpus = extract_texts_from_directory(
    '/content/canonical-latinLit',
    caesar_codes,
    lang='la'
)
print(f"César: {len(caesar_corpus)} fichiers chargés")

# 2. Créer les fenêtres pour César
caesar_windows = {'lexical': [], 'thematic': [], 'structural': []}
for file_id, levels_dict in caesar_corpus.items():
    for level, segments in levels_dict.items():
        for seg in segments:
            if seg.strip():
                caesar_windows[level].append((seg, file_id))

print(f"\nFenêtres César:")
for level in ['lexical', 'thematic', 'structural']:
    print(f"  {level}: {len(caesar_windows[level])} segments")

# 3. Encoder César
print("\nEncodage César...")
caesar_embeddings = {}
for level in ['lexical', 'thematic', 'structural']:
    texts = [seg for seg, _ in caesar_windows[level]]
    print(f"  Niveau {level}...")
    embeddings = compute_embeddings_sphilberta(texts)
    caesar_embeddings[level] = embeddings
    print(f"    Shape: {embeddings.shape}")

# 4. Analyser les correspondances avec César
print("\n=== COMPARAISON : TRAGIQUES GRECS → CÉSAR ===\n")

results_caesar = {}
for greek_author in ['sophocles', 'aeschylus', 'euripides']:
    results_caesar[greek_author] = {}

    for level in ['lexical', 'thematic', 'structural']:
        # Mêmes paramètres que pour Sénèque
        if level == 'lexical':
            threshold = 0.82
            top_k = 5
        elif level == 'thematic':
            threshold = 0.78
            top_k = 10
        else:
            threshold = 0.73
            top_k = 15

        echoes, sim_matrix = find_semantic_echoes(
            greek_embeddings[greek_author],
            caesar_embeddings[level],
            threshold=threshold,
            top_k=top_k
        )

        results_caesar[greek_author][level] = {
            'echoes': echoes,
            'echo_count': len(echoes),
            'avg_similarity': np.mean([e['scores'].mean() for e in echoes]) if echoes else 0
        }

        print(f"{greek_author} → César {level}:")
        print(f"  Échos: {len(echoes)}/300")
        print(f"  Similarité: {results_caesar[greek_author][level]['avg_similarity']:.3f}")

# 5. Comparaison Sénèque vs César
print("\n=== COMPARAISON SÉNÈQUE vs CÉSAR ===\n")

for level in ['lexical', 'thematic', 'structural']:
    print(f"\nNiveau {level.upper()}:")
    print("  Similarités moyennes avec les tragiques grecs:")
    print("           Sophocle  Eschyle  Euripide")

    # Sénèque
    sen_scores = [results_adjusted[a][level]['avg_similarity'] for a in ['sophocles', 'aeschylus', 'euripides']]
    print(f"  Sénèque:  {sen_scores[0]:.3f}    {sen_scores[1]:.3f}    {sen_scores[2]:.3f}")

    # César
    cae_scores = [results_caesar[a][level]['avg_similarity'] for a in ['sophocles', 'aeschylus', 'euripides']]
    print(f"  César:    {cae_scores[0]:.3f}    {cae_scores[1]:.3f}    {cae_scores[2]:.3f}")

    # Différence
    diffs = [s-c for s,c in zip(sen_scores, cae_scores)]
    print(f"  Δ (S-C):  {diffs[0]:+.3f}   {diffs[1]:+.3f}   {diffs[2]:+.3f}")

    # Test statistique
    if all(len(results_caesar[a][level]['echoes']) > 0 for a in ['sophocles', 'aeschylus', 'euripides']):
        # Comparer Sénèque vs César pour chaque auteur grec
        for greek in ['sophocles', 'aeschylus', 'euripides']:
            sen_echo_scores = [e['scores'].max() for e in results_adjusted[greek][level]['echoes']]
            cae_echo_scores = [e['scores'].max() for e in results_caesar[greek][level]['echoes']]

            if sen_echo_scores and cae_echo_scores:
                stat, p = stats.mannwhitneyu(sen_echo_scores, cae_echo_scores)
                if p < 0.05:
                    print(f"    {greek}: Sénèque > César (p={p:.4f})")

# Visualisations des résultats

In [ ]:
# Collecter les données depuis votre analyse existante
tragedy_names = {
        'phi001': 'Hercules Furens',
        'phi002': 'Troades',
        'phi003': 'Phoenissae',
        'phi004': 'Medea',
        'phi005': 'Phaedra',
        'phi006': 'Oedipus',
        'phi007': 'Agamemnon',
        'phi008': 'Thyestes',
        'phi009': 'Hercules Oetaeus'
    }

influence_matrix = {}
for level in ['lexical', 'thematic', 'structural']:
    tragedy_scores = {name: {'sophocles': 0, 'aeschylus': 0, 'euripides': 0}
                     for name in tragedy_names.values()}

    for greek_author in ['sophocles', 'aeschylus', 'euripides']:
        echoes = results_adjusted[greek_author][level]['echoes']

        for echo in echoes:
            for latin_idx, score in zip(echo['latin_indices'], echo['scores']):
                file_id = seneca_windows[level][latin_idx][1]
                for code, name in tragedy_names.items():
                    if code in file_id:
                        if tragedy_scores[name][greek_author] == 0:
                            tragedy_scores[name][greek_author] = []
                        tragedy_scores[name][greek_author].append(score)

    # Moyenner
    for tragedy in tragedy_names.values():
        for author in ['sophocles', 'aeschylus', 'euripides']:
            if isinstance(tragedy_scores[tragedy][author], list):
                tragedy_scores[tragedy][author] = np.mean(tragedy_scores[tragedy][author])

    influence_matrix[level] = tragedy_scores

# Créer matrice pour heatmap (moyenne des 3 niveaux)
matrix = []
for tragedy in tragedy_names.values():
    row = []
    for author in ['sophocles', 'aeschylus', 'euripides']:
        scores = [influence_matrix[level][tragedy][author]
                 for level in ['lexical', 'thematic', 'structural']]
        row.append(np.mean([s for s in scores if s > 0]) if any(s > 0 for s in scores) else 0)
    matrix.append(row)

# Visualisation
plt.figure(figsize=(8, 6))
sns.heatmap(matrix,
            annot=True,
            fmt='.3f',
            xticklabels=['Sophocle', 'Eschyle', 'Euripide'],
            yticklabels=list(tragedy_names.values()),
            cmap='YlOrRd',
            vmin=0.76, vmax=0.82)
plt.title('Influence des tragiques grecs sur chaque tragédie de Sénèque')
plt.tight_layout()
plt.show()

In [ ]:
# Même code de préparation des données...

fig, ax = plt.subplots(figsize=(14, 7))

labels = [w['label'] for w in top_15]
scores = [w['mean_score'] for w in top_15]
counts = [w['count'] for w in top_15]

colors = ['#FF6B6B' if w['author'] == 'sophocles' else
          '#4ECDC4' if w['author'] == 'aeschylus' else
          '#45B7D1' for w in top_15]

bars = ax.bar(range(len(labels)), scores, color=colors, alpha=0.8)

# IMPORTANT: Ajuster l'échelle Y pour mieux voir les différences
min_score = min(scores) - 0.01
max_score = max(scores) + 0.01
ax.set_ylim(min_score, max_score)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('Similarité moyenne')
ax.set_title('Top 15 œuvres grecques influençant Sénèque')

# Afficher les comptes avec une taille de police adaptée
for bar, count in zip(bars, counts):
    height = bar.get_height()
    # Si les comptes sont très grands, les formater
    count_label = f'{count}' if count < 1000 else f'{count/1000:.1f}k'
    ax.text(bar.get_x() + bar.get_width()/2., height,
            count_label, ha='center', va='bottom', fontsize=7)

# Ajouter une ligne horizontale pour la moyenne
mean_all = np.mean(scores)
ax.axhline(y=mean_all, color='gray', linestyle='--', alpha=0.5, label=f'Moyenne: {mean_all:.3f}')

plt.legend(handles=[Patch(facecolor='#FF6B6B', label='Sophocle'),
                    Patch(facecolor='#4ECDC4', label='Eschyle'),
                    Patch(facecolor='#45B7D1', label='Euripide')])
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

# Construire la matrice de flux : œuvre grecque -> tragédie Sénèque
flows = {}
for greek_author in ['sophocles', 'aeschylus', 'euripides']:
    for level in ['lexical', 'thematic', 'structural']:
        echoes = results_adjusted[greek_author][level]['echoes']

        for echo in echoes:
            greek_idx = echo['greek_idx']
            greek_file = representative_segments[greek_author][greek_idx]['file_id']

            # Extraire le code de l'œuvre grecque
            if '.' in greek_file:
                work_code = greek_file.split('.')[1]
            else:
                work_code = greek_file

            greek_key = f"{greek_author[:4].upper()}-{work_code}"

            # Pour chaque correspondance latine
            for latin_idx, score in zip(echo['latin_indices'], echo['scores']):
                latin_file = seneca_windows[level][latin_idx][1]

                # Identifier la tragédie de Sénèque
                for code, sen_name in tragedy_names.items():
                    if code in latin_file:
                        key = (greek_key, sen_name)
                        if key not in flows:
                            flows[key] = []
                        flows[key].append(score)
                        break

# Agréger les flux
aggregated_flows = {}
for (source, target), scores in flows.items():
    aggregated_flows[(source, target)] = {
        'mean': np.mean(scores),
        'count': len(scores)
    }

# Option 1: Réseau bipartite
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# Créer le graphe
G = nx.Graph()
greek_nodes = list(set([k[0] for k in aggregated_flows.keys()]))
seneca_nodes = list(set([k[1] for k in aggregated_flows.keys()]))

# Ajouter les nœuds
G.add_nodes_from(greek_nodes, bipartite=0)
G.add_nodes_from(seneca_nodes, bipartite=1)

# Ajouter les arêtes pondérées (seulement les plus fortes)
threshold = sorted([v['mean'] for v in aggregated_flows.values()], reverse=True)[50]  # Top 50 connexions
for (source, target), data in aggregated_flows.items():
    if data['mean'] >= threshold:
        G.add_edge(source, target, weight=data['mean'], count=data['count'])

# Positionner les nœuds
pos = {}
# Œuvres grecques à gauche
for i, node in enumerate(sorted(greek_nodes)):
    pos[node] = (0, i * 0.5)
# Tragédies Sénèque à droite
for i, node in enumerate(sorted(seneca_nodes)):
    pos[node] = (4, i * 1.2)

# Dessiner
nx.draw_networkx_nodes(G, pos, nodelist=greek_nodes, node_color='lightblue',
                      node_size=1000, ax=ax1)
nx.draw_networkx_nodes(G, pos, nodelist=seneca_nodes, node_color='lightcoral',
                      node_size=1500, ax=ax1)

# Dessiner les arêtes avec épaisseur variable
edges = G.edges(data=True)
weights = [e[2]['weight'] for e in edges]
min_w, max_w = min(weights), max(weights)
widths = [1 + 10 * (w - min_w) / (max_w - min_w) for w in weights]

nx.draw_networkx_edges(G, pos, width=widths, alpha=0.5, ax=ax1)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax1)

ax1.set_title('Réseau d\'influence: Œuvres grecques → Tragédies de Sénèque', fontsize=14, fontweight='bold')
ax1.axis('off')

# Option 2: Heatmap détaillée
# Créer une matrice pour la heatmap
greek_works = sorted(list(set([k[0] for k in aggregated_flows.keys()])))
seneca_works = sorted(tragedy_names.values())

matrix = np.zeros((len(greek_works), len(seneca_works)))
for i, greek in enumerate(greek_works):
    for j, seneca in enumerate(seneca_works):
        if (greek, seneca) in aggregated_flows:
            matrix[i, j] = aggregated_flows[(greek, seneca)]['mean']

# Filtrer pour ne garder que les œuvres grecques avec influence significative
row_sums = matrix.sum(axis=1)
top_indices = np.argsort(row_sums)[-20:]  # Top 20 œuvres grecques
matrix_filtered = matrix[top_indices]
greek_works_filtered = [greek_works[i] for i in top_indices]

# Créer la heatmap
im = ax2.imshow(matrix_filtered, cmap='YlOrRd', aspect='auto', vmin=0.75, vmax=0.82)
ax2.set_xticks(range(len(seneca_works)))
ax2.set_xticklabels([s.split()[0] for s in seneca_works], rotation=45, ha='right')
ax2.set_yticks(range(len(greek_works_filtered)))
ax2.set_yticklabels(greek_works_filtered, fontsize=8)
ax2.set_title('Matrice d\'influence détaillée (Top 20 œuvres grecques)', fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax2, label='Similarité moyenne')

plt.tight_layout()
plt.show()

# Statistiques résumées
print("\n=== CONNEXIONS LES PLUS FORTES ===\n")
top_connections = sorted(aggregated_flows.items(), key=lambda x: x[1]['mean'], reverse=True)[:10]
for (greek, seneca), data in top_connections:
    print(f"{greek:20} → {seneca:20} : {data['mean']:.3f} ({data['count']} échos)")

In [ ]:
def analyze_specific_pair(greek_author, greek_work_code, seneca_tragedy_code,
                         representative_segments, seneca_windows,
                         greek_embeddings, seneca_embeddings):
    """Analyse détaillée entre une œuvre grecque et une tragédie de Sénèque"""

    # Filtrer les segments grecs pour l'œuvre spécifique
    greek_indices = []
    greek_texts = []
    for i, seg_data in enumerate(representative_segments[greek_author]):
        if greek_work_code in seg_data['file_id']:
            greek_indices.append(i)
            greek_texts.append(seg_data['segment'])

    if not greek_indices:
        print(f"Aucun segment trouvé pour {greek_author} - {greek_work_code}")
        return None

    # Filtrer les segments de Sénèque pour la tragédie spécifique
    seneca_data = {'lexical': [], 'thematic': [], 'structural': []}
    seneca_indices = {'lexical': [], 'thematic': [], 'structural': []}

    for level in ['lexical', 'thematic', 'structural']:
        for i, (text, file_id) in enumerate(seneca_windows[level]):
            if seneca_tragedy_code in file_id:
                seneca_data[level].append(text)
                seneca_indices[level].append(i)

    # Calculer les similarités pour chaque niveau
    results_by_level = {}

    for level in ['lexical', 'thematic', 'structural']:
        if not seneca_indices[level]:
            continue

        # Extraire les embeddings pertinents
        greek_emb_subset = greek_embeddings[greek_author][greek_indices]
        seneca_emb_subset = seneca_embeddings[level][seneca_indices[level]]

        # Calculer la matrice de similarité
        similarities = np.dot(greek_emb_subset, seneca_emb_subset.T) / (
            np.linalg.norm(greek_emb_subset, axis=1)[:, np.newaxis] *
            np.linalg.norm(seneca_emb_subset, axis=1)[np.newaxis, :]
        )

        # Trouver les meilleures correspondances
        top_matches = []
        for i in range(len(greek_indices)):
            best_j = np.argmax(similarities[i])
            score = similarities[i, best_j]
            if score > 0.7:  # Seuil de similarité
                top_matches.append({
                    'greek_text': greek_texts[i],
                    'seneca_text': seneca_data[level][best_j],
                    'score': score
                })

        results_by_level[level] = {
            'similarities': similarities,
            'top_matches': sorted(top_matches, key=lambda x: x['score'], reverse=True)[:10],
            'mean_sim': np.mean(similarities),
            'max_sim': np.max(similarities),
            'std_sim': np.std(similarities)
        }

    return results_by_level

# Analyser Euripide tlg008 (Suppliantes) vs Sénèque Troades
results_pair = analyze_specific_pair(
    'euripides', 'tlg008', 'phi002',
    representative_segments, seneca_windows,
    greek_embeddings, seneca_embeddings
)

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Titre principal
fig.suptitle('Analyse des similarités: Euripide Suppliantes (tlg008) ↔ Sénèque Troades',
             fontsize=16, fontweight='bold')

# 1. Distributions des similarités par niveau
ax1 = axes[0, 0]
levels = ['lexical', 'thematic', 'structural']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for i, level in enumerate(levels):
    if level in results_pair:
        similarities = results_pair[level]['similarities'].flatten()
        ax1.hist(similarities, bins=30, alpha=0.6, label=f'{level.capitalize()} (μ={np.mean(similarities):.3f})',
                color=colors[i], density=True)

ax1.set_xlabel('Score de similarité')
ax1.set_ylabel('Densité')
ax1.set_title('Distribution des similarités par niveau')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Top scores par niveau
ax2 = axes[0, 1]
bar_data = []
bar_labels = []
bar_colors = []

for i, level in enumerate(levels):
    if level in results_pair:
        top_scores = [m['score'] for m in results_pair[level]['top_matches'][:5]]
        bar_data.extend(top_scores)
        bar_labels.extend([f'{level[:3]}-{j+1}' for j in range(len(top_scores))])
        bar_colors.extend([colors[i]] * len(top_scores))

ax2.bar(range(len(bar_data)), bar_data, color=bar_colors, alpha=0.8)
ax2.set_xticks(range(len(bar_data)))
ax2.set_xticklabels(bar_labels, rotation=45, ha='right')
ax2.set_ylabel('Score de similarité')
ax2.set_title('Top 5 correspondances par niveau')
ax2.set_ylim(0.7, max(bar_data) + 0.02)
ax2.grid(axis='y', alpha=0.3)

# 3. Heatmap d'un niveau (thématique)
ax3 = axes[1, 0]
if 'thematic' in results_pair:
    sim_matrix = results_pair['thematic']['similarities']
    # Limiter la taille pour la lisibilité
    max_size = min(50, sim_matrix.shape[0], sim_matrix.shape[1])
    im = ax3.imshow(sim_matrix[:max_size, :max_size], cmap='YlOrRd', aspect='auto', vmin=0.6, vmax=0.85)
    ax3.set_xlabel('Segments Sénèque Troades')
    ax3.set_ylabel('Segments Euripide Suppliantes')
    ax3.set_title(f'Matrice de similarité thématique ({max_size}x{max_size} segments)')
    plt.colorbar(im, ax=ax3)
else:
    ax3.text(0.5, 0.5, 'Pas de données thématiques', ha='center', va='center')
    ax3.axis('off')

# 4. Statistiques comparatives
ax4 = axes[1, 1]
stats_data = []
for level in levels:
    if level in results_pair:
        stats_data.append([
            results_pair[level]['mean_sim'],
            results_pair[level]['max_sim'],
            results_pair[level]['std_sim']
        ])
    else:
        stats_data.append([0, 0, 0])

stats_data = np.array(stats_data).T
x = np.arange(len(levels))
width = 0.25

bars1 = ax4.bar(x - width, stats_data[0], width, label='Moyenne', color='steelblue', alpha=0.8)
bars2 = ax4.bar(x, stats_data[1], width, label='Maximum', color='coral', alpha=0.8)
bars3 = ax4.bar(x + width, stats_data[2], width, label='Écart-type', color='green', alpha=0.8)

ax4.set_xlabel('Niveau d\'analyse')
ax4.set_ylabel('Valeur')
ax4.set_title('Statistiques de similarité')
ax4.set_xticks(x)
ax4.set_xticklabels([l.capitalize() for l in levels])
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Afficher les meilleures correspondances textuelles
print("\n=== MEILLEURES CORRESPONDANCES TEXTUELLES ===\n")
for level in levels:
    if level in results_pair and results_pair[level]['top_matches']:
        print(f"\n{level.upper()} - Top 3:")
        print("-" * 80)
        for i, match in enumerate(results_pair[level]['top_matches'][:3], 1):
            print(f"\n[{i}] Score: {match['score']:.3f}")
            print(f"GREC:  {match['greek_text'][:100]}...")
            print(f"LATIN: {match['seneca_text'][:100]}...")

# Code de recherche de correspondance de phrase unique

Ce petit code permet de chercher une phrase de Sénèque particulière et de voir quel est l'extrait des auteurs tragiques le plus proche.

In [ ]:
# Chercher la phrase de Sénèque dans le corpus d'Euripide
def find_closest_euripides_to_seneca_phrase(phrase_seneca, representative_segments, greek_embeddings, model):
    """
    Trouve le segment d'Euripide le plus proche d'une phrase spécifique de Sénèque
    """
    # Encoder la phrase de Sénèque
    seneca_embedding = model.encode([phrase_seneca], convert_to_tensor=True, device=device)
    seneca_embedding = seneca_embedding.cpu().numpy()

    # Récupérer les embeddings d'Euripide
    euripides_embeddings = greek_embeddings['euripides']

    # Calculer les similarités cosinus
    similarities = np.dot(euripides_embeddings, seneca_embedding.T).flatten() / (
        np.linalg.norm(euripides_embeddings, axis=1) * np.linalg.norm(seneca_embedding)
    )

    # Trouver les top 5 correspondances
    top_indices = np.argsort(similarities)[-5:][::-1]

    results = []
    for idx in top_indices:
        results.append({
            'index': idx,
            'greek_text': representative_segments['euripides'][idx]['segment'],
            'file_id': representative_segments['euripides'][idx]['file_id'],
            'similarity': similarities[idx]
        })

    return results

# Exécuter la recherche
phrase_seneca = "curae leves loquuntur ingentes stupent"
print(f"Recherche des correspondances pour : '{phrase_seneca}'")
print("="*80)

matches = find_closest_euripides_to_seneca_phrase(
    phrase_seneca,
    representative_segments,
    greek_embeddings,
    model
)

for i, match in enumerate(matches, 1):
    print(f"\n[{i}] Score: {match['similarity']:.4f}")
    print(f"Œuvre: {match['file_id']}")
    print(f"Grec: {match['greek_text']}")

# Vérifier aussi dans tout le corpus d'Euripide (pas seulement les segments représentatifs)
print("\n" + "="*80)
print("Recherche élargie dans tout le corpus d'Euripide...")

# Récupérer tous les segments d'Euripide
all_euripides_segments = []
for seg, file_id in tragedian_windows['euripides']:
    if seg.strip():
        all_euripides_segments.append((seg, file_id))

# Prendre un échantillon (car encoder tout serait trop long)
sample_size = min(1000, len(all_euripides_segments))
import random
sample_indices = random.sample(range(len(all_euripides_segments)), sample_size)
sample_segments = [all_euripides_segments[i] for i in sample_indices]

# Encoder les segments échantillonnés
print(f"Encodage de {sample_size} segments d'Euripide...")
sample_texts = [seg for seg, _ in sample_segments]
sample_embeddings = compute_embeddings_sphilberta(sample_texts)

# Comparer avec la phrase de Sénèque
seneca_embedding = model.encode([phrase_seneca], convert_to_tensor=True, device=device).cpu().numpy()

similarities = np.dot(sample_embeddings, seneca_embedding.T).flatten() / (
    np.linalg.norm(sample_embeddings, axis=1) * np.linalg.norm(seneca_embedding)
)

# Top 3 des meilleures correspondances
top_indices = np.argsort(similarities)[-3:][::-1]

print(f"\nMeilleures correspondances sur {sample_size} segments:")
for idx in top_indices:
    print(f"\nScore: {similarities[idx]:.4f}")
    print(f"Fichier: {sample_segments[idx][1]}")
    print(f"Grec: {sample_segments[idx][0]}")

# Visualisations test

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt
from IPython.display import display

def _search_in_dict(d, needle):
    """Cherche `needle` (insensible à la casse) dans toutes les valeurs str du dict `d`."""
    if needle is None:
        return True
    n = str(needle).lower()
    for v in d.values():
        try:
            if n in str(v).lower():
                return True
        except Exception:
            pass
    return False

def select_greek_indices(author='euripides', work_hint=None):
    """
    Renvoie la liste des indices de segments grecs pour `author` dont les métadonnées
    contiennent `work_hint` (ex. 'tlg008' ou 'Suppliantes'). Si `work_hint` est None
    ou introuvable, renvoie *tous* les segments disponibles pour l'auteur.
    """
    items = representative_segments[author]
    if work_hint is None:
        return list(range(len(items)))
    idx = [i for i, d in enumerate(items) if isinstance(d, dict) and _search_in_dict(d, work_hint)]
    if not idx:  # fallback: tous
        print(f"⚠️ Aucun segment grec trouvé pour '{work_hint}'. Utilisation de tous les segments de {author}.")
        idx = list(range(len(items)))
    return idx

def select_seneca_indices(level='thematic', work_hint='troades'):
    """
    Renvoie la liste des indices latins au niveau `level` dont le file_id contient `work_hint`
    (ex. 'troades', 'thyestes', ou un code comme 'phi010'). Si introuvable, renvoie tous.
    """
    ids = [fid for _, fid in seneca_windows[level]]
    if work_hint is None:
        return list(range(len(ids)))
    idx = [i for i, fid in enumerate(ids) if work_hint.lower() in str(fid).lower()]
    if not idx:
        print(f"⚠️ Aucun segment latin trouvé pour '{work_hint}' au niveau '{level}'. Utilisation de tous les segments.")
        idx = list(range(len(ids)))
    return idx

def oeuvre_heatmap(author='euripides', greek_work='tlg008', seneca_work='troades',
                   level='thematic', topk=5, cmap='YlOrRd'):
    """
    Exemple d'appel:
        oeuvre_heatmap(author='euripides', greek_work='tlg008', seneca_work='troades', level='thematic')
    Nécessite: representative_segments, greek_embeddings, seneca_windows, seneca_embeddings.
    """
    gi = select_greek_indices(author, greek_work)
    li = select_seneca_indices(level, seneca_work)

    G = greek_embeddings[author][gi]
    L = seneca_embeddings[level][li]
    if G.size == 0 or L.size == 0:
        raise ValueError("Embeddings vides — vérifiez vos filtres.")

    # Similarités cosinus
    sims = np.dot(G, L.T) / (
        np.linalg.norm(G, axis=1)[:, None] * np.linalg.norm(L, axis=1)[None, :]
    )

    # Heatmap segment×segment
    plt.figure(figsize=(7,5))
    plt.imshow(sims, aspect='auto', origin='lower', cmap=cmap)
    plt.colorbar(label="Similarité")
    plt.xlabel(f"Segments Sénèque ({seneca_work})")
    plt.ylabel(f"Segments {author.capitalize()} ({greek_work})")
    plt.title(f"Carte de chaleur — {author.capitalize()} ({greek_work}) ↔ Sénèque ({seneca_work}) [{level}]")
    plt.tight_layout()
    plt.show()

    # Top-k max & min (indices i, j + valeurs)
    flat = sims.ravel()
    order_max = np.argsort(flat)[-topk:][::-1]
    order_min = np.argsort(flat)[:topk]

    def ij(idx):
        return divmod(int(idx), sims.shape[1])

    tops = [(ij(k)[0], ij(k)[1], float(flat[k])) for k in order_max]
    lows = [(ij(k)[0], ij(k)[1], float(flat[k])) for k in order_min]

    df_tops = pd.DataFrame(tops, columns=['greek_idx','latin_idx','similarity'])
    df_lows = pd.DataFrame(lows, columns=['greek_idx','latin_idx','similarity'])

    display(df_tops.style.set_caption("Top-5 correspondances les plus fortes"))
    display(df_lows.style.set_caption("Top-5 correspondances les plus faibles"))

    return sims, gi, li  # utile si tu veux réutiliser la matrice ensuite

In [ ]:
from collections import Counter
import pandas as pd

def seneca_works(level='structural'):
    fids = [fid for _, fid in seneca_windows[level]]
    cnt = Counter(fids)
    return (pd.DataFrame({'fid': list(cnt.keys()), 'n_segments': [cnt[f] for f in cnt]})
              .sort_values('fid', ignore_index=True))

# Exemple : voir tous les fid disponibles en "structural"
seneca_works('structural')

In [ ]:
oeuvre_heatmap(author='sophocles',
               greek_work='tlg004',
               seneca_work='phi1017.phi006',  # Oedipus
               level='structural')